<a href="https://colab.research.google.com/github/Inderjeet-singh01/Shine-Dezign/blob/main/AI_Movie_Advisor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project : AI Movie Advisor**

* **Core Recommendation Logic:** Developed a system that recommends similar movies by analyzing item metadata, ensuring users find content closely related to their viewing history.

* **NLP-Driven Feature Engineering:** Leveraged Natural Language Processing (NLP) techniques to process and clean unstructured text data, creating a unified "Content Soup" from descriptions, genres, and cast details.

* **Text Vectorization (TF-IDF):** Applied NLP vectorization using TF-IDF to convert movie text into mathematical vectors, allowing the model to understand the importance of specific keywords over common words.

* **Similarity Computation:** Utilized Cosine Similarity to calculate the distance between movie vectors, enabling the engine to rank and suggest the top 10 most relevant titles in real-time.



In [1]:
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
netflix_df= pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ML/Netflix  Recommendation/netflix_titles.csv')
netflix_df.tail(2)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."
8806,s8807,Movie,Zubaan,Mozez Singh,"Vicky Kaushal, Sarah-Jane Dias, Raaghav Chanan...",India,"March 2, 2019",2015,TV-14,111 min,"Dramas, International Movies, Music & Musicals",A scrappy but poor boy worms his way into a ty...


In [3]:
netflix_df.isnull().sum()

,0
show_id,0
type,0
title,0
director,2634
cast,825
country,831
date_added,10
release_year,0
rating,4
duration,3


In [4]:
## Analysing IMDB ratings to get top rated movies on Netflix

imdb_df= pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ML/Netflix  Recommendation/imdb.csv')
imdb_df = imdb_df[['title', 'imdb_score','imdb_votes']]    #selecting features
imdb_df.head(3)

,title,imdb_score,imdb_votes
0,Taxi Driver,8.3,795222.0
1,Monty Python and the Holy Grail,8.2,530877.0
2,Life of Brian,8.0,392419.0


In [5]:
## Performing inner join on the imdb_df and netflix_df to get the content that has both ratings on IMDB and are available on Netflix.

joint_data = imdb_df.merge(netflix_df, left_on='title', right_on='title', how='inner')
joint_data=joint_data.sort_values(by='imdb_score', ascending=False)
joint_data.head(2)


,title,imdb_score,imdb_votes,show_id,type,director,cast,country,date_added,release_year,rating,duration,listed_in,description
391,Khawatir,9.6,3046.0,s369,TV Show,NaN,NaN,NaN,"July 30, 2021",2009,TV-14,1 Season,"International TV Shows, Reality TV",Saudi media personality Ahmad Al Shugairi trav...
132,Breaking Bad,9.5,1727694.0,s5941,TV Show,NaN,"Bryan Cranston, Aaron Paul, Anna Gunn, Dean No...",United States,"August 2, 2013",2013,TV-MA,5 Seasons,"Crime TV Shows, TV Dramas, TV Thrillers",A high school chemistry teacher dying of cance...


In [6]:
# Top 10 highest rating Movies/TV shows

top_10= joint_data[['title','release_year','imdb_score']].head(10)
top_10 = top_10.reset_index(drop=True)
top_10.index = range(1, len(top_10)+1)
print('************* Top 10 Highest Rating Movies/TV Shows *************')
top_10

************* Top 10 Highest Rating Movies/TV Shows *************


,title,release_year,imdb_score
1,Khawatir,2009,9.6
2,Breaking Bad,2013,9.5
3,Our Planet,2019,9.3
4,Kota Factory,2021,9.3
5,Avatar: The Last Airbender,2007,9.3
6,Reply 1988,2015,9.2
7,My Mister,2018,9.2
8,The Last Dance,2020,9.1
9,Attack on Titan,2013,9.0
10,Leah Remini: Scientology and the Aftermath,2018,9.0


In [7]:
joint_data.isnull().sum()

,0
title,0
imdb_score,0
imdb_votes,5
show_id,0
type,0
director,1362
cast,352
country,282
date_added,0
release_year,0


In [8]:
## Handling missing value in country column

most_common = joint_data['country'].mode()[0]   #most repeated country
print('Most Repeated Country is :',most_common)

joint_data['country'] = joint_data['country'].fillna(most_common)  #filling null country value with most repeated country
joint_data.isnull().sum()

Most Repeated Country is : United States


,0
title,0
imdb_score,0
imdb_votes,5
show_id,0
type,0
director,1362
cast,352
country,0
date_added,0
release_year,0


In [9]:
joint_data.shape

(3790, 14)

In [10]:
#removing null values
joint_data = joint_data.dropna(subset=['imdb_votes','duration'])

# filling with empty space
joint_data['cast']= joint_data['cast'].fillna("")

# Droping unwanted columns
joint_data= joint_data.drop(['show_id','date_added','director','rating','duration'], axis=1)

#Removing rows with duplicate titles
joint_data = joint_data.drop_duplicates(subset=['title'])

#to reset index
joint_data= joint_data.reset_index(drop=True)

#final shape of dataset
joint_data.shape

(3741, 9)

In [11]:
#Final database
joint_data.head(2)

,title,imdb_score,imdb_votes,type,cast,country,release_year,listed_in,description
0,Khawatir,9.6,3046.0,TV Show,,United States,2009,"International TV Shows, Reality TV",Saudi media personality Ahmad Al Shugairi trav...
1,Breaking Bad,9.5,1727694.0,TV Show,"Bryan Cranston, Aaron Paul, Anna Gunn, Dean No...",United States,2013,"Crime TV Shows, TV Dramas, TV Thrillers",A high school chemistry teacher dying of cance...


# **Recommendation System (Content Based)**

In [12]:
#to convert text to vector
from sklearn.feature_extraction.text import TfidfVectorizer

#removing stopwords
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2))

#used for cosine simmilarity, creating a dataframe with title and index
indices= pd.Series(joint_data.index, index=joint_data['title']).drop_duplicates()
indices

,0
title,
Khawatir,0
Breaking Bad,1
Our Planet,2
Kota Factory,3
Avatar: The Last Airbender,4
...,...
Hajwala: The Missing Engine,3736
Kyaa Kool Hain Hum 3,3737
Racket Boys,3738


In [13]:
# Joining all the text in new 'tags' column, so we can apply vectorization easliy

joint_data['tags'] = joint_data['cast'] +' '+ joint_data['country'] +' '+ joint_data['listed_in']+' '+ joint_data['type']+' '+ joint_data['description']
joint_data.head()

,title,imdb_score,imdb_votes,type,cast,country,release_year,listed_in,description,tags
0,Khawatir,9.6,3046.0,TV Show,,United States,2009,"International TV Shows, Reality TV",Saudi media personality Ahmad Al Shugairi trav...,"United States International TV Shows, Reality..."
1,Breaking Bad,9.5,1727694.0,TV Show,"Bryan Cranston, Aaron Paul, Anna Gunn, Dean No...",United States,2013,"Crime TV Shows, TV Dramas, TV Thrillers",A high school chemistry teacher dying of cance...,"Bryan Cranston, Aaron Paul, Anna Gunn, Dean No..."
2,Our Planet,9.3,41386.0,TV Show,David Attenborough,"United States, United Kingdom",2019,"Docuseries, Science & Nature TV",Experience our planet's natural beauty and exa...,"David Attenborough United States, United Kingd..."
3,Kota Factory,9.3,66985.0,TV Show,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K..."
4,Avatar: The Last Airbender,9.3,297336.0,TV Show,"Zach Tyler, Mae Whitman, Jack De Sena, Dee Bra...",United States,2007,"Classic & Cult TV, Kids' TV, TV Action & Adven...",Siblings Katara and Sokka wake young Aang from...,"Zach Tyler, Mae Whitman, Jack De Sena, Dee Bra..."


In [14]:
#converting 'tags' column to vector matrix

tfidf_matrix = tfidf.fit_transform(joint_data['tags'])

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

ref_movie=[]      #to store reference movie name

def Recomend(title, n=10):
  if title not in indices:
    return ['Movie not found']

  idx = indices[title]  #Reference movie
  ref_movie.append(title)

  sim_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()  #Comparing all movies with referen[ce movie

  similar_idx = sim_score.argsort()[::-1][1:n+1]  #Getting top n similar movies

  return joint_data['title'].iloc[similar_idx]

In [16]:
## Getting recomended movies

recomendations= Recomend('Breaking Bad')

print(f" 🎬 Recommended Movies like : '{ref_movie[0]}' \n")

for i, movie in enumerate(recomendations, start=1):
    print(f"{i}. {movie}")

ref_movie.clear()

 🎬 Recommended Movies like : 'Breaking Bad' 

1. Better Call Saul
2. Girlfriend's Day
3. El Camino: A Breaking Bad Movie
4. Dear White People
5. Dare Me
6. Murder Among the Mormons
7. Age of Rebellion
8. Cocaine Cowboys: The Kings of Miami
9. American Vandal
10. Ozark


### Above Model Details
Raw dataset-> Data cleaning-> joint_data-> TF-IDF vectorize-> tfidf_matrix-> cosine_similarity-> Recomendations